# XOPT - OR-LIBRARY p-MEDIAN WITH MIP

It solves every OR-Library p-median instance with `sklearn_extra.cluster.KMedoids` and reports runtime and gap to the published best-known value, and uses the `KMedoids` solution as a feasible warm start for the exact `python-mip` formulation and optionally adds the heuristic objective as a cutoff constraint.

### SETUP

In [1]:
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

Importing the libraries:

In [2]:
import gc
import os
import sys

import numpy  as np
import pandas as pd

from IPython.display       import display
from sklearn_extra.cluster import KMedoids
from time                  import perf_counter

from mip import xsum, minimize, Model, BINARY, OptimizationStatus

In [3]:
from lib.paths     import find_project_root, \
                          instance_sort_key

from lib.instances import list_orlibrary_instances     , \
                          apply_instance_selection     , \
                          load_best_known_costs_to_dict

from lib.metrics   import gap_to_reference_percent

from lib.graph     import read_orlibrary_graph    , \
                          all_pairs_shortest_paths

from lib.mip       import extract_open_facilities, \
                          compute_solution_cost  , \
                          compute_assignments

from lib.utils     import finite_or_none        , \
                          normalize_number      , \
                          parse_optional_int_env

Find the project root directory:

In [4]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


### EXPERIMENT CONFIGURATION

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / 'instances'
PMEDOPT_PATH  = INSTANCES_DIR / 'pmedopt.txt'
OUTPUT_DIR    = PROJECT_ROOT  / 'notebooks' / 'experiments_sbpo' / 'artifacts'

KMEDOIDS_OUTPUT_PATH   = OUTPUT_DIR / 'kmedoids_results.csv'
WARM_START_OUTPUT_PATH = OUTPUT_DIR / 'mip_warm_start_results.csv'

TIME_LIMIT_SECONDS = parse_optional_int_env('PMED_TIME_LIMIT_SECONDS') or 120
MAX_INSTANCES      = parse_optional_int_env('PMED_MAX_INSTANCES'     )

INSTANCE_FILTER    = os.getenv('PMED_INSTANCE_FILTER')

KMEDOIDS_RANDOM_STATE = parse_optional_int_env('PMED_KMEDOIDS_RANDOM_STATE') or 0
KMEDOIDS_MAX_ITER     = parse_optional_int_env('PMED_KMEDOIDS_MAX_ITER'    ) or 5
KMEDOIDS_METHOD       = os.getenv('PMED_KMEDOIDS_METHOD', 'pam'      )
KMEDOIDS_INIT         = os.getenv('PMED_KMEDOIDS_INIT'  , 'heuristic')

APPLY_HEURISTIC_CUTOFF = os.getenv('PMED_APPLY_HEURISTIC_CUTOFF', '1') != '0'


CANONICAL_INSTANCE_NAMES = [
    f'pmed{i}.txt' for i in range(1, 41)
]

ALL_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES     = apply_instance_selection(
    ALL_INSTANCE_NAMES,
    pattern=INSTANCE_FILTER,
    limit  =MAX_INSTANCES  ,
)
BEST_KNOWN_COSTS   = load_best_known_costs_to_dict(PMEDOPT_PATH)


if not INSTANCE_NAMES:
    raise FileNotFoundError(
        f'No p-median instances were selected from {INSTANCES_DIR}.'
    )

In [6]:
print(f'Instances directory  : {INSTANCES_DIR}')
print(f'Instances discovered : {len(ALL_INSTANCE_NAMES)}')
print(f'Instances selected   : {len(INSTANCE_NAMES    )}')

Instances directory  : /home/rei-luisinho/xopt/instances
Instances discovered : 40
Instances selected   : 40


In [7]:
print(f'Time limit per instance (s)     : {TIME_LIMIT_SECONDS    }')
print(f'KMedoids random_state           : {KMEDOIDS_RANDOM_STATE }')
print(f'KMedoids max_iter               : {KMEDOIDS_MAX_ITER     }')
print(f'Heuristic cutoff applied to MIP : {APPLY_HEURISTIC_CUTOFF}')
print(f'KMedoids method / init          : {KMEDOIDS_METHOD} / {KMEDOIDS_INIT}')

Time limit per instance (s)     : 120
KMedoids random_state           : 0
KMedoids max_iter               : 5
Heuristic cutoff applied to MIP : True
KMedoids method / init          : pam / heuristic


In [8]:
missing_canonical = sorted(set(CANONICAL_INSTANCE_NAMES) - set(ALL_INSTANCE_NAMES      ), key=instance_sort_key)
extra_instances   = sorted(set(ALL_INSTANCE_NAMES      ) - set(CANONICAL_INSTANCE_NAMES), key=instance_sort_key)
missing_optima    = sorted(set(ALL_INSTANCE_NAMES      ) - set(BEST_KNOWN_COSTS        ), key=instance_sort_key)

if missing_canonical:
    print(f'Missing canonical OR-Library instances: {missing_canonical}')
else:
    print('All 40 canonical OR-Library instances are present.')

if extra_instances:
    print(f'Additional instances discovered    : {extra_instances}')

if missing_optima:
    print(f'Instances without best-known value : {missing_optima}')

print()
print('Instances to solve:')
print(', '.join(name.removesuffix('.txt') for name in INSTANCE_NAMES))

All 40 canonical OR-Library instances are present.

Instances to solve:
pmed1, pmed2, pmed3, pmed4, pmed5, pmed6, pmed7, pmed8, pmed9, pmed10, pmed11, pmed12, pmed13, pmed14, pmed15, pmed16, pmed17, pmed18, pmed19, pmed20, pmed21, pmed22, pmed23, pmed24, pmed25, pmed26, pmed27, pmed28, pmed29, pmed30, pmed31, pmed32, pmed33, pmed34, pmed35, pmed36, pmed37, pmed38, pmed39, pmed40


### MIP + WARM START

In [9]:
def build_pmedian_model(
    distances             : np.ndarray,
    p                     : int,
    objective_upper_bound : float | int | None = None,
) -> tuple[Model, list[list], list]:
    if distances.ndim != 2:
        raise ValueError('Distances must be a 2D array.')

    n_rows, n_cols = distances.shape
    if n_rows != n_cols:
        raise ValueError(
            'This implementation assumes a square distance matrix.'
        )

    n = n_rows
    if not (1 <= p <= n):
        raise ValueError('P must satisfy 1 <= p <= n.')

    if np.any(distances < 0):
        raise ValueError('Distances must be nonnegative.')

    model = Model(solver_name='CBC')

    model.verbose = 0

    y = [
        model.add_var(
            var_type=BINARY, name=f'y_{j}'
        )
        for j in range(n)
    ]

    x: list[list]  = []
    row_cost_terms = []

    for i in range(n):
        x_row = [
            model.add_var(
                var_type=BINARY, name=f'x_{i}_{j}'
            )
            for j in range(n)
        ]
        x.append(x_row)

        model.add_constr(
            xsum(x_row[j] for j in range(n)) == 1, name=f'assign_{i}'
        )

        for j in range(n):
            model.add_constr(x_row[j] <= y[j], name=f'link_{i}_{j}')

        row_cost_terms.append(
            xsum(float(distances[i, j]) * x_row[j] for j in range(n))
        )

    model.add_constr(xsum(y[j] for j in range(n)) == p, name='select_p')

    objective_expr = xsum(row_cost_terms)

    objective_upper_bound = finite_or_none(
        objective_upper_bound
    )
    if objective_upper_bound is not None:
        model.add_constr(
            objective_expr <= objective_upper_bound,
            name='heuristic_upper_bound'           ,
        )

    model.objective = minimize(objective_expr)

    return model, x, y


def run_kmedoids(
    distances : np.ndarray,
    p         : int       ,
) -> dict[str, object]:
    solve_start = perf_counter()

    estimator = KMedoids(
        n_clusters   =p,
        metric       ='precomputed'        ,
        method       =KMEDOIDS_METHOD      ,
        init         =KMEDOIDS_INIT        ,
        max_iter     =KMEDOIDS_MAX_ITER    ,
        random_state =KMEDOIDS_RANDOM_STATE,
    )

    estimator.fit(distances)

    solve_seconds = perf_counter() - solve_start

    open_facilities_zero_based = sorted(
        int(index)
        for index in estimator.medoid_indices_.tolist()
    )

    objective_value = compute_solution_cost(
        distances, open_facilities_zero_based
    )

    inertia_value = finite_or_none(
        getattr(
            estimator, 'inertia_', None
        )
    )

    if objective_value is not None and \
       inertia_value   is not None:
        if abs(objective_value - inertia_value) > 1e-6:
            raise ValueError(
                'KMedoids inertia and validated objective do not match: '
                f'{inertia_value} vs {objective_value}.'
            )

    return {
        'objective_value'            : objective_value,
        'solve_seconds'              : solve_seconds  ,
        'n_iter'                     : getattr(estimator, 'n_iter_', None),
        'open_facilities_zero_based' : open_facilities_zero_based         ,
    }


def apply_mip_start(
    model           : Model     ,
    x               : list[list],
    y               : list      ,
    open_facilities : list[int] ,
    distances       : np.ndarray,
) -> None:
    assignments = compute_assignments(
        distances, open_facilities
    )

    start    = []
    open_set = set(open_facilities)

    for j, variable in enumerate(y):
        start.append(
            (
                variable,
                1.0 if j in open_set else 0.0
            )
        )

    for i, x_row in enumerate(x):
        chosen_facility = assignments[i]

        for j, variable in enumerate(x_row):
            start.append(
                (
                    variable,
                    1.0 if j == chosen_facility else 0.0
                )
            )

    model.start = start

In [10]:
def solve_instance_with_kmedoids_warm_start(
    instance_name      : str                     ,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    instance_path = INSTANCES_DIR / instance_name
    graph         = read_orlibrary_graph(instance_path)

    preprocess_start   = perf_counter()
    distances          = all_pairs_shortest_paths(graph['adjacency'])
    preprocess_seconds = perf_counter() - preprocess_start

    best_known_value         = finite_or_none(BEST_KNOWN_COSTS.get(instance_name))
    kmedoids_result          = run_kmedoids  (distances, graph['p'])
    kmedoids_objective_value = finite_or_none(kmedoids_result['objective_value'])

    if kmedoids_objective_value is not None and best_known_value is not None:
        if kmedoids_objective_value + 1e-6 < best_known_value:
            raise ValueError(
                'KMedoids objective is below the published best-known value: '
                f'{kmedoids_objective_value} < {best_known_value}.'
            )

    build_start = perf_counter()
    model, x, y = build_pmedian_model(
        distances ,
        graph['p'],
        objective_upper_bound=(
            kmedoids_objective_value if APPLY_HEURISTIC_CUTOFF else None
        ),
    )
    build_seconds = perf_counter() - build_start

    apply_mip_start(
        model,
        x    ,
        y    ,
        kmedoids_result['open_facilities_zero_based'],
        distances                                    ,
    )

    solve_start   = perf_counter  ()
    status        = model.optimize(max_seconds=time_limit_seconds)
    solve_seconds = perf_counter  () - solve_start

    has_incumbent = status in {
        OptimizationStatus.OPTIMAL ,
        OptimizationStatus.FEASIBLE,
    }

    solver_objective_value = finite_or_none(
        model.objective_value if has_incumbent else None
    )

    open_facilities_zero_based: list[int] = []

    validated_objective_value = None

    if has_incumbent:
        open_facilities_zero_based = extract_open_facilities(y)

        if len(open_facilities_zero_based) != graph['p']:
            raise ValueError(
                f'Expected {graph["p"]} open facilities, but recovered {len(open_facilities_zero_based)}.'
            )

        validated_objective_value = compute_solution_cost(
            distances, open_facilities_zero_based
        )

        if solver_objective_value    is not None and \
           validated_objective_value is not None:
            if abs(solver_objective_value - validated_objective_value) > 1e-4:
                raise ValueError(
                    'Solver objective and validated objective do not match: '
                    f'{solver_objective_value} vs {validated_objective_value}.'
                )

        if validated_objective_value is not None and \
           best_known_value          is not None:
            if validated_objective_value + 1e-6 < best_known_value:
                raise ValueError(
                    'Validated objective is below the published best-known value: '
                    f'{validated_objective_value} < {best_known_value}.'
                )

    objective_value = (
        validated_objective_value
        if   validated_objective_value is not None
        else solver_objective_value
    )

    kmedoids_open_facilities = [
        facility + 1
        for facility in kmedoids_result['open_facilities_zero_based']
    ]
    open_facilities = [
        facility + 1
        for facility in open_facilities_zero_based
    ]

    result = {
        'instance' : instance_name,
        'n'        : graph['n'],
        'm'        : graph['m'],
        'p'        : graph['p'],

        'best_known_value' : normalize_number(best_known_value),

        'kmedoids_open_facilities_count'      : len     (         kmedoids_open_facilities ),
        'kmedoids_open_facilities'            : ' '.join(map(str, kmedoids_open_facilities)),
        'kmedoids_solve_seconds'              : normalize_number(kmedoids_result['solve_seconds']),
        'kmedoids_n_iter'                     : normalize_number(kmedoids_result['n_iter'       ]),
        'kmedoids_objective_value'            : normalize_number(kmedoids_objective_value        ),
        'kmedoids_gap_to_best_known_percent'  : normalize_number(
            gap_to_reference_percent(
                kmedoids_objective_value,
                best_known_value        ,
            )
        ),

        'warm_start_applied'      : True,
        'heuristic_cutoff_applied': APPLY_HEURISTIC_CUTOFF and kmedoids_objective_value is not None,

        'status'                   : getattr(status, 'name', str(status)),
        'objective_value'          : normalize_number(objective_value   ),
        'gap_to_best_known_percent': normalize_number(
            gap_to_reference_percent(
                objective_value ,
                best_known_value,
            )
        ),

        'matches_best_known' : (
            objective_value  is not None and
            best_known_value is not None and
            abs(objective_value - best_known_value) < 1e-6
        ),

        'preprocess_seconds'  : normalize_number(preprocess_seconds           ),
        'model_build_seconds' : normalize_number(build_seconds                ),
        'solve_seconds'       : normalize_number(solve_seconds                ),
        'mip_stage_seconds'   : normalize_number(build_seconds + solve_seconds),
        'total_seconds'       : normalize_number(
            preprocess_seconds               +
            kmedoids_result['solve_seconds'] +
            build_seconds                    +
            solve_seconds
        ),

        'open_facilities_count' : len(open_facilities               ),
        'open_facilities'       : ' '.join(map(str, open_facilities)),

        'error': '',
    }

    del model
    del x
    del y
    del distances

    gc.collect()

    return result


def attempt_solve_instance_with_kmedoids_warm_start(
    instance_name      : str                     ,
    time_limit_seconds : int = TIME_LIMIT_SECONDS,
) -> dict[str, object]:
    try:
        return solve_instance_with_kmedoids_warm_start(
            instance_name, time_limit_seconds=time_limit_seconds
        )
    except Exception as exc:
        gc.collect()

        return {
            'instance' : instance_name,
            'n'        : None,
            'm'        : None,
            'p'        : None,

            'best_known_value' : normalize_number(BEST_KNOWN_COSTS.get(instance_name)),

            'kmedoids_n_iter'                    : None,
            'kmedoids_objective_value'           : None,
            'kmedoids_gap_to_best_known_percent' : None,
            'kmedoids_solve_seconds'             : None,
            'kmedoids_open_facilities_count'     : 0 ,
            'kmedoids_open_facilities'           : '',

            'warm_start_applied'       : False  ,
            'heuristic_cutoff_applied' : False  ,
            'status'                   : 'ERROR',
            'objective_value'          : None   ,
            'solver_gap_percent'       : None   ,
            'gap_to_best_known_percent': None   ,
            'matches_best_known'       : False  ,

            'preprocess_seconds'  : None,
            'model_build_seconds' : None,
            'solve_seconds'       : None,
            'mip_stage_seconds'   : None,
            'total_seconds'       : None,

            'open_facilities_count' : 0 ,
            'open_facilities'       : '',

            'error': str(exc),
        }

### APPLY

In [11]:
RESULTS = []

for index, instance_name in enumerate(INSTANCE_NAMES, start=1):
    print(f'[{index:02d}/{len(INSTANCE_NAMES):02d}] Solving {instance_name}...')

    result = attempt_solve_instance_with_kmedoids_warm_start(
        instance_name, time_limit_seconds=TIME_LIMIT_SECONDS
    )

    RESULTS.append(result)

    print(
        f"  kmedoids_obj={result['kmedoids_objective_value'          ]},"
        f" kmedoids_gap={ result['kmedoids_gap_to_best_known_percent']},"
        f" mip_status={   result['status'                            ]},"
        f" mip_obj={      result['objective_value'                   ]},"
        f" mip_gap={      result['gap_to_best_known_percent'         ]},"
        f" total_seconds={result['total_seconds'                     ]}"
    )

    if result['error']:
        print(f"  error={result['error']}")

RESULTS_DF = pd.DataFrame(RESULTS)

[01/40] Solving pmed1.txt...
  kmedoids_obj=5819, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=5819, mip_gap=0, total_seconds=0.8699
[02/40] Solving pmed2.txt...
  kmedoids_obj=4768, kmedoids_gap=16.4916, mip_status=OPTIMAL, mip_obj=4093, mip_gap=0, total_seconds=1.778
[03/40] Solving pmed3.txt...
  kmedoids_obj=4693, kmedoids_gap=10.4235, mip_status=OPTIMAL, mip_obj=4250, mip_gap=0, total_seconds=1.8159
[04/40] Solving pmed4.txt...
  kmedoids_obj=3921, kmedoids_gap=29.2353, mip_status=OPTIMAL, mip_obj=3034, mip_gap=0, total_seconds=1.4201
[05/40] Solving pmed5.txt...
  kmedoids_obj=2347, kmedoids_gap=73.2103, mip_status=OPTIMAL, mip_obj=1355, mip_gap=0, total_seconds=1.4373
[06/40] Solving pmed6.txt...
  kmedoids_obj=7824, kmedoids_gap=0, mip_status=OPTIMAL, mip_obj=7824, mip_gap=0, total_seconds=19.4931
[07/40] Solving pmed7.txt...
  kmedoids_obj=6093, kmedoids_gap=8.2046, mip_status=OPTIMAL, mip_obj=5631, mip_gap=0, total_seconds=12.4863
[08/40] Solving pmed8.txt...
  kmedoids_obj=60

Saving the results:

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KMEDOIDS_RESULTS_DF = RESULTS_DF[
    [
        'instance',
        'n'       ,
        'm'       ,
        'p'       ,
        'best_known_value',

        'kmedoids_n_iter'                   ,
        'kmedoids_objective_value'          ,
        'kmedoids_gap_to_best_known_percent',
        'kmedoids_solve_seconds'            ,
        'kmedoids_open_facilities_count'    ,
        'kmedoids_open_facilities'          ,

        'error',
    ]
].copy()

KMEDOIDS_RESULTS_DF.to_csv(KMEDOIDS_OUTPUT_PATH  , index=False)
RESULTS_DF         .to_csv(WARM_START_OUTPUT_PATH, index=False)

print(f'KMedoids results saved to      : {KMEDOIDS_OUTPUT_PATH  }')
print(f'Warm-start MIP results saved to: {WARM_START_OUTPUT_PATH}')

KMedoids results saved to      : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/kmedoids_results.csv
Warm-start MIP results saved to: /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/mip_warm_start_results.csv


Displaying the results:

In [14]:
def print_table_summary(
    dataframe      : pd.DataFrame,
    label          : str,
    objective_col  : str,
    gap_col        : str,
    time_col       : str,
) -> None:
    solved_count  = int( dataframe[objective_col].notna().sum())
    matches_count = int((dataframe[gap_col      ].fillna(np.inf).abs() < 1e-6).sum())

    mean_time     = dataframe[time_col].dropna().mean()
    mean_gap      = dataframe[gap_col ].dropna().mean()

    print(label)
    print(f'  solved instances            : {solved_count }/{len(dataframe)}')
    print(f'  matches best-known          : {matches_count}/{len(dataframe)}')
    print(f'  mean time (s)               : {normalize_number(mean_time)}')
    print(f'  mean gap to best-known (%)  : {normalize_number(mean_gap )}')
    print()


print_table_summary(
    KMEDOIDS_RESULTS_DF,
    label        ='KMedoids summary:'       ,
    objective_col='kmedoids_objective_value',
    gap_col ='kmedoids_gap_to_best_known_percent',
    time_col='kmedoids_solve_seconds'            ,
)

print_table_summary(
    RESULTS_DF,
    label        ='Warm-start exact MIP summary:',
    objective_col='objective_value'              ,
    gap_col ='gap_to_best_known_percent',
    time_col='total_seconds'            ,
)

WARM_START_DISPLAY_DF = RESULTS_DF[
    [
        'instance',

        'best_known_value'                  ,
        'kmedoids_objective_value'          ,
        'kmedoids_gap_to_best_known_percent',
        'kmedoids_solve_seconds'            ,

        'objective_value'          ,
        'gap_to_best_known_percent',

        'solve_seconds',
        'total_seconds',
        'status'       ,
    ]
].copy()

KMedoids summary:
  solved instances            : 40/40
  matches best-known          : 6/40
  mean time (s)               : 0.1288
  mean gap to best-known (%)  : 37.7801

Warm-start exact MIP summary:
  solved instances            : 35/40
  matches best-known          : 30/40
  mean time (s)               : 93.6577
  mean gap to best-known (%)  : 1.5338



Approach overview:

In [15]:
APPROACH_SUMMARY_DF = pd.DataFrame(
    [
        {
            'approach'                       : 'sklearn-extra KMedoids',
            'instances'                      : len(KMEDOIDS_RESULTS_DF),
            'mean_runtime_seconds'           : normalize_number(KMEDOIDS_RESULTS_DF['kmedoids_solve_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(KMEDOIDS_RESULTS_DF['kmedoids_gap_to_best_known_percent'].dropna().mean()),
        },
        {
            'approach'                       : 'Warm-start MIP (solve only)',
            'instances'                      : len(RESULTS_DF              ),
            'mean_runtime_seconds'           : normalize_number(RESULTS_DF['solve_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(RESULTS_DF['gap_to_best_known_percent'].dropna().mean()),
        },
        {
            'approach'                       : 'Warm-start MIP (full workflow)',
            'instances'                      : len(RESULTS_DF                 ),
            'mean_runtime_seconds'           : normalize_number(RESULTS_DF['total_seconds'            ].dropna().mean()),
            'mean_gap_to_best_known_percent' : normalize_number(RESULTS_DF['gap_to_best_known_percent'].dropna().mean()),
        },
    ]
)

display(APPROACH_SUMMARY_DF)

,approach,instances,mean_runtime_seconds,mean_gap_to_best_known_percent
0,sklearn-extra KMedoids,40,0.1288,37.7801
1,Warm-start MIP (solve only),40,89.8552,1.5338
2,Warm-start MIP (full workflow),40,93.6577,1.5338


KMedoids per-instance results:

In [16]:
display(KMEDOIDS_RESULTS_DF)

,instance,n,m,p,best_known_value,kmedoids_n_iter,kmedoids_objective_value,kmedoids_gap_to_best_known_percent,kmedoids_solve_seconds,kmedoids_open_facilities_count,kmedoids_open_facilities,error
0,pmed1.txt,100,200,5,5819,4,5819,0.0000,0.0055,5,7 13 65 91 99,
1,pmed2.txt,100,200,10,4093,4,4768,16.4916,0.0043,10,2 12 16 23 24 27 52 55 58 67,
2,pmed3.txt,100,200,10,4250,4,4693,10.4235,0.0026,10,8 10 12 14 26 36 55 77 96 99,
3,pmed4.txt,100,200,20,3034,4,3921,29.2353,0.0040,20,7 8 12 22 26 31 34 38 43 44 48 55 73 76 77 78 ...,
4,pmed5.txt,100,200,33,1355,4,2347,73.2103,0.0039,33,5 10 14 15 16 19 22 24 27 35 36 38 40 44 47 49...,
5,pmed6.txt,200,800,5,7824,4,7824,0.0000,0.0063,5,16 86 101 111 126,
6,pmed7.txt,200,800,10,5631,4,6093,8.2046,0.0082,10,10 40 131 138 140 142 160 186 191 199,
7,pmed8.txt,200,800,20,4445,4,6039,35.8605,0.0123,20,9 13 31 32 35 84 88 91 94 114 115 121 130 141 ...,
8,pmed9.txt,200,800,40,2734,4,4353,59.2173,0.0156,40,2 8 12 14 15 17 18 27 30 39 44 45 46 65 66 67 ...,
9,pmed10.txt,200,800,67,1255,4,2673,112.9880,0.0167,67,6 7 12 20 21 28 29 32 37 38 39 40 41 45 47 53 ...,


Warm-start exact MIP per-instance results:

In [17]:
display(WARM_START_DISPLAY_DF)

,instance,best_known_value,kmedoids_objective_value,kmedoids_gap_to_best_known_percent,kmedoids_solve_seconds,objective_value,gap_to_best_known_percent,solve_seconds,total_seconds,status
0,pmed1.txt,5819,5819,0.0000,0.0055,5819.0,0.0000,0.7118,0.8699,OPTIMAL
1,pmed2.txt,4093,4768,16.4916,0.0043,4093.0,0.0000,1.6665,1.7780,OPTIMAL
2,pmed3.txt,4250,4693,10.4235,0.0026,4250.0,0.0000,1.7203,1.8159,OPTIMAL
3,pmed4.txt,3034,3921,29.2353,0.0040,3034.0,0.0000,1.3274,1.4201,OPTIMAL
4,pmed5.txt,1355,2347,73.2103,0.0039,1355.0,0.0000,1.3427,1.4373,OPTIMAL
5,pmed6.txt,7824,7824,0.0000,0.0063,7824.0,0.0000,19.0779,19.4931,OPTIMAL
6,pmed7.txt,5631,6093,8.2046,0.0082,5631.0,0.0000,12.1138,12.4863,OPTIMAL
7,pmed8.txt,4445,6039,35.8605,0.0123,4445.0,0.0000,12.2022,12.5873,OPTIMAL
8,pmed9.txt,2734,4353,59.2173,0.0156,2734.0,0.0000,12.3419,12.7233,OPTIMAL
9,pmed10.txt,1255,2673,112.9880,0.0167,1255.0,0.0000,12.5334,12.9179,OPTIMAL
